# CRIVO — Destilação v2: 87 mil conversas novas, professor maior e juiz (piloto primeiro)

O que mudou da v1 (`experimentos/destilacao_20261014/RESULTADO_PARTE2.md`, `PLANO_V2.md`, `professor_v2.py`):
- **falas novas**, escritas pelo professor a partir de 133 temas × 20 intenções × 7 jeitos de escrever. Nenhuma fala que já está na fase 5 (treino, dev, blinds, frases reais) entra: nenhuma fala do treino ganha uma segunda resposta;
- **conversa de dois turnos**: o professor escreve também a reação da pessoa e responde de novo;
- **professor maior** (o primeiro que existir e couber: Qwen3-32B-AWQ, Qwen3-14B-AWQ, Qwen3-14B; licença Apache 2.0);
- **juiz de outra família** (Mistral Small 24B ou Phi-4) dá nota de 1 a 5; **só entra nota 4 ou 5**;
- **o juiz é testado**: a cada rodada ele recebe 60 respostas trocadas de propósito (resposta de outra conversa). Se aprovar mais de 10% delas, o notebook **para**: juiz que aprova lixo não serve;
- **sem repetição**: a mesma resposta entra no máximo 3 vezes, e nenhum começo de resposta passa de 1% do caderno;
- **sem fatos do professor**; 15% das falas novas ficam guardadas como prova do juiz e nunca treinam.

**Meta: 87.000 conversas aprovadas** para o treino. O notebook roda em rodadas até chegar nela (ou até acabar a grade, e aí diz quanto faltou, sem baixar a exigência).

**`PILOTO = True` primeiro** (uns 15 a 25 minutos): poucas falas, e a última célula mostra 30 conversas aprovadas e 10 reprovadas. **Mande o print e leia você também.** Com o seu ok, `PILOTO = False` faz o caderno inteiro (umas 5 a 7 horas; se a sessão cair, rode de novo e ele continua do pedaço em que parou).

Regras (AGENTS.md): os modelos de terceiros só rodam aqui, os pesos ficam no disco do Colab e são apagados; no Drive fica só o caderno, marcado com modelo, revisão, data e licença.

In [ ]:
# Célula 0 — GPU e Drive
import os, sys, json, gzip, shutil, subprocess, hashlib, random, time
from pathlib import Path
from collections import Counter, defaultdict
PILOTO = True                    # primeiro True; com o ok do criador, False
os.environ["HF_HOME"] = "/content/hf"
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("ERRO: sem GPU. Ambiente de execução → A100.")
GPU, MEM = r.stdout.strip().split(", ")
print("GPU:", GPU, MEM, "MB", flush=True)
if int(MEM) < 38000:
    raise SystemExit("ERRO: precisa de A100 (40 GB ou mais).")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "dados-fase6b").is_dir()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei a pasta CRIVO com dados-fase6b. Rode na conta hr87676620.")
SAIDA = CRIVO / "dados-professor-v2" / ("piloto" if PILOTO else "completo")
SAIDA.mkdir(parents=True, exist_ok=True)
print("saída:", SAIDA, flush=True)

In [ ]:
# Célula 1 — código, vLLM e as falas que não podem repetir
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)
EXP = REPO / "experimentos" / "destilacao_20261014"
for p in (str(REPO), str(EXP)):
    if p not in sys.path:
        sys.path.insert(0, p)
for m in ("professor", "professor_v2"):
    sys.modules.pop(m, None)
import professor as P, professor_v2 as V
EXP5 = REPO / "experimentos" / "fase5_conversa_20261013"
subprocess.run([sys.executable, str(EXP5 / "gerar_v4.py")], cwd=EXP5, check=True)
PROIBIDAS = set()
for f in ("dados_fase5.json.gz", "dados_fase5_v2.json.gz", "dados_fase5_v3.json.gz", "dados_fase5_v4.json.gz"):
    for casos in json.loads(gzip.open(EXP5 / f).read()).values():
        if isinstance(casos, list):
            for c in casos:
                PROIBIDAS.add(P.norm(c.get("mensagem", "")))
                PROIBIDAS.update(P.norm(h) for h in c.get("historico", []))
PROIBIDAS.discard("")
print("commit", COMMIT[:10], "| professor_v2", V.VERSAO, "| falas da fase 5 que não podem repetir:", len(PROIBIDAS), flush=True)

PROFESSORES = ["Qwen/Qwen3-32B-AWQ", "Qwen/Qwen3-14B-AWQ", "Qwen/Qwen3-14B"]
JUIZES = ["RedHatAI/Mistral-Small-3.2-24B-Instruct-2506-quantized.w4a16",
          "RedHatAI/Mistral-Small-3.1-24B-Instruct-2503-quantized.w4a16", "microsoft/phi-4"]
ESCOLHIDO = {}

def escrever(nome, itens):
    p = SAIDA / nome
    with gzip.open(p, "wt", encoding="utf-8") as f:
        for x in itens:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    return p

def liberar_gpu():
    """Encerra processos que ainda seguram a GPU (um modelo travado de antes sobrevive até a sessão reiniciar)."""
    r = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], capture_output=True, text=True)
    pids = [int(x) for x in r.stdout.split() if x.strip().isdigit() and int(x) != os.getpid()]
    for pid in pids:
        try:
            os.kill(pid, 9)
        except ProcessLookupError:
            pass
    subprocess.run(["pkill", "-9", "-f", "rodar_modelo.py"], capture_output=True)
    if pids:
        time.sleep(5)
        print(f"   GPU liberada (encerrei {len(pids)} processo(s) que sobraram)", flush=True)
    livre = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
    print(f"   GPU livre: {livre} MB", flush=True)

liberar_gpu()

def rodar(papel, nome, itens):
    """Roda o primeiro modelo da lista que existir, tiver a licença certa e couber; devolve {id: (texto, fim)}."""
    entrada, saida = escrever(nome + "_entrada.jsonl.gz", itens), SAIDA / (nome + ".jsonl.gz")
    lista = [ESCOLHIDO[papel]] if papel in ESCOLHIDO else (PROFESSORES if papel == "professor" else JUIZES)
    limite = None if papel == "professor" else 900 + 0.5 * len(itens)   # juiz travado: tenta o próximo
    for modelo in lista:
        if not (saida.is_file()):
            liberar_gpu()
        cmd = [sys.executable, "-u", str(EXP / "rodar_modelo.py"), "--modelo", modelo, "--papel", papel,
               "--entrada", str(entrada), "--saida", str(saida)]
        import threading
        estourou = threading.Event()
        with subprocess.Popen(cmd, cwd=EXP, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                              start_new_session=True) as proc:
            # o vLLM abre processos filhos; matar o grupo inteiro devolve a memória da GPU
            vigia = threading.Timer(limite, lambda: (estourou.set(), os.killpg(proc.pid, 9))) if limite else None
            if vigia:
                vigia.start()
            for linha in proc.stdout:   # mostra o que o modelo está fazendo (o Colab não mostra o processo filho sozinho)
                if any(x in linha for x in ("pedaço", "pedidos", "respostas", "não achei", "licença", "não carregou",
                                            "já feito", "não monta", "Error", "error", "Traceback")):
                    print("   ", linha.rstrip()[-200:], flush=True)
            proc.wait()
            if vigia:
                vigia.cancel()
        codigo = 3 if estourou.is_set() else proc.returncode
        if estourou.is_set():
            print(f"   {modelo}: passou de {limite / 60:.0f} min sem terminar, encerrado; tentando o próximo da lista", flush=True)
        shutil.rmtree("/content/hf", ignore_errors=True) if codigo != 0 else None
        if codigo == 0:
            ESCOLHIDO[papel] = modelo
            break
        if codigo != 3:
            raise SystemExit(f"ERRO rodando {modelo} (código {codigo}). Mande o print.")
    else:
        raise SystemExit(f"ERRO: nenhum {papel} da lista funcionou: {lista}. Mande o print.")
    with gzip.open(saida, "rt", encoding="utf-8") as f:
        return {b["id"]: (b["texto"], b["fim"]) for b in map(json.loads, f)}

In [ ]:
# Célula 2 — uma rodada: falas novas → resposta → reação da pessoa → resposta → juiz (com controles)
VISTAS = set(PROIBIDAS)

def responder(nome, perguntas):
    itens = [{"id": p["id"], "mensagens": P.mensagens(p), "temperatura": 0.7,
              "max_tokens": P.MAX_TOKENS_LONGA if P.longa(p) else P.MAX_TOKENS_CURTA} for p in perguntas]
    R = rodar("professor", nome, itens)
    out = []
    for p in perguntas:
        if p["id"] in R:
            texto, fim = R[p["id"]]
            c = V.conferir(p, texto)
            if fim == "length":
                c["ok"] = False; c["motivos"].append("cortada")
            out.append((p, c))
    return out

def rodada(nome, pedidos):
    R = rodar("professor", nome + "_a_falas", [{"id": p["id"], "mensagens": p["mensagens"], "max_tokens": 400, "temperatura": 0.9} for p in pedidos])
    falas, rel = V.falas_novas([(p, R[p["id"]][0]) for p in pedidos if p["id"] in R], VISTAS)
    VISTAS.update(P.norm(f["turnos"][0]["texto"]) for f in falas)
    t1 = responder(nome + "_b_respostas1", falas)
    bons = [(p, c) for p, c in t1 if c["ok"]]
    R = rodar("professor", nome + "_c_proximas", [{"id": p["id"], "mensagens": V.pedir_proxima(p, c["texto"]), "max_tokens": 80,
                                                   "temperatura": 0.8} for p, c in bons])
    segundas = []
    for p, c in bons:
        if p["id"] in R:
            q = V.segundo_turno(p, c["texto"], R[p["id"]][0], VISTAS)
            if q:
                VISTAS.add(P.norm(q["turnos"][-1]["texto"])); segundas.append(q)
    t2 = responder(nome + "_d_respostas2", segundas)
    shutil.rmtree("/content/hf", ignore_errors=True)            # o professor sai do disco antes do juiz
    todas = [(p, c) for p, c in t1 + t2 if c["ok"]]
    ctrl = V.controles(todas, random.Random(nome))
    R = rodar("juiz", nome + "_e_notas", [{"id": p["id"], "mensagens": V.pedir_nota(p, c["texto"]), "max_tokens": 12, "temperatura": 0.0}
                                          for p, c in todas] + [{"id": p["id"], "mensagens": V.pedir_nota(p, r), "max_tokens": 12,
                                                                 "temperatura": 0.0} for p, r in ctrl])
    shutil.rmtree("/content/hf", ignore_errors=True)
    notas = {k: V.ler_nota(t) for k, (t, _) in R.items()}
    ok_juiz, aprovou_lixo = V.juiz_confiavel([notas.get(p["id"]) for p, _ in ctrl])
    rel.update({"pedidos": len(pedidos), "turno1": len(t1), "turno1_ok": sum(c["ok"] for _, c in t1), "turno2": len(t2),
                "turno2_ok": sum(c["ok"] for _, c in t2), "motivos": dict(Counter(m for _, c in t1 + t2 for m in c["motivos"])),
                "notas": {str(k): v for k, v in Counter(notas.get(p["id"]) for p, _ in todas).items()},
                "controles": len(ctrl), "controles_aprovados": aprovou_lixo, "juiz_confiavel": ok_juiz})
    print(f"\n== rodada {nome}: {json.dumps(rel, ensure_ascii=False)}", flush=True)
    if not ok_juiz:
        raise SystemExit(f"PAROU: o juiz aprovou {100 * aprovou_lixo:.0f}% das respostas trocadas de propósito. Mande o print.")
    return [(p, c, notas.get(p["id"])) for p, c in todas], rel

In [ ]:
# Célula 3 — as rodadas até a meta (no piloto, uma rodada pequena)
ORDEM = V.pedidos(random.Random(V.SEED))          # a mesma ordem sempre; o piloto usa os 40 primeiros
APROVADAS, RODADAS = [], []
if PILOTO:
    a, rel = rodada("piloto", ORDEM[:40]); APROVADAS += a; RODADAS.append(rel)
else:
    MP = json.loads((SAIDA.parent / "piloto" / "manifesto.json").read_text())
    por_pedido = max(0.5, MP["treino_por_pedido"])
    inicio, k = 40, 0
    while inicio < len(ORDEM) and k < 8:
        treino_ate_agora = sum(1 for p, c, n in APROVADAS if n is not None and n >= V.NOTA_MINIMA and p["destino"] == "treino")
        falta = V.META - treino_ate_agora * 0.9          # ~10% sai na tirada de repetição
        if falta <= 0:
            break
        n = min(len(ORDEM) - inicio, max(500, int(falta / por_pedido * 1.15)))
        a, rel = rodada(f"r{k}", ORDEM[inicio:inicio + n]); APROVADAS += a; RODADAS.append(rel)
        inicio, k = inicio + n, k + 1
        por_pedido = max(0.5, sum(1 for p, c, n_ in a if n_ is not None and n_ >= V.NOTA_MINIMA and p["destino"] == "treino") / n)
print("rodadas:", len(RODADAS), flush=True)

In [ ]:
# Célula 4 — o caderno: só nota 4 ou 5, sem repetição, até a meta; a prova separada
INFO = {papel: json.loads(next(SAIDA.glob(f"*_{s}_modelo.json")).read_text()) for papel, s in (("professor", "b_respostas1"), ("juiz", "e_notas"))}
prof = {"rotulo": "v2", **{k: INFO["professor"][k] for k in ("modelo", "revisao", "licenca", "data")}}
TREINO, PROVA, FORA = [], [], []
for p, c, nota in APROVADAS:
    e = {**P.exemplo(p, c, prof), "nota_juiz": nota, "tema": p["tema"]}
    (FORA if nota is None or nota < V.NOTA_MINIMA else PROVA if p["destino"] == "prova" else TREINO).append(e)
TREINO, REL_DIV = V.diversificar(TREINO)
TREINO.sort(key=lambda e: (-e["nota_juiz"], e["chave"]))
SOBRA = max(0, len(TREINO) - V.META)
TREINO = TREINO[:V.META] if not PILOTO else TREINO          # passou da meta: ficam as de nota maior
bruto = json.dumps(TREINO, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
with gzip.GzipFile(SAIDA / "dados_professor_v2.json.gz", "wb", mtime=0) as f:
    f.write(bruto)
(SAIDA / "prova_juiz.json").write_text(json.dumps([{"id": e["chave"], "turnos": e["turnos"][:-1], "resposta_professor": e["turnos"][-1]["texto"]}
                                                   for e in PROVA], ensure_ascii=False, indent=0))
pedidos_usados = sum(r["pedidos"] for r in RODADAS)
MAN = {"protocolo": "destilacao_v2_" + ("piloto" if PILOTO else "completo"), "commit": COMMIT, "professor_v2": V.VERSAO,
       "professor": INFO["professor"], "juiz": INFO["juiz"], "rodadas": RODADAS, "meta": V.META, "treino": len(TREINO),
       "prova": len(PROVA), "reprovadas_pelo_juiz": len(FORA), "tirada_de_repeticao": REL_DIV, "sobra_cortada": SOBRA,
       "treino_por_pedido": round((len(TREINO) + sum(REL_DIV.values())) / max(1, pedidos_usados), 3),
       "notas_no_treino": {str(k): v for k, v in Counter(e["nota_juiz"] for e in TREINO).items()},
       "dois_turnos_no_treino": sum(len(e["turnos"]) > 2 for e in TREINO), "dados_sha256": hashlib.sha256(bruto).hexdigest(),
       "aviso": "dados gerados por modelos de terceiros (professor) e notas de outro (juiz); não são falas humanas (AGENTS.md)"}
(SAIDA / "manifesto.json").write_text(json.dumps(MAN, ensure_ascii=False, indent=1))
print(json.dumps({k: MAN[k] for k in ("treino", "meta", "prova", "reprovadas_pelo_juiz", "tirada_de_repeticao", "notas_no_treino",
                                      "dois_turnos_no_treino", "treino_por_pedido")}, ensure_ascii=False), flush=True)
print("professor:", INFO["professor"]["modelo"], "| juiz:", INFO["juiz"]["modelo"], "| controles aprovados pelo juiz:",
      [r["controles_aprovados"] for r in RODADAS], flush=True)
if not PILOTO and len(TREINO) < V.META:
    print(f"AVISO: ficaram {len(TREINO)} de {V.META}; a grade acabou. A exigência não foi baixada.", flush=True)

def mostrar(e):
    for t in e["turnos"]:
        print(("   PESSOA: " if t["papel"] == "usuario" else "   CRIVO:  ") + t["texto"][:300], flush=True)
rr = random.Random(5)
print("\n===== 30 conversas aprovadas (nota 4 ou 5) =====", flush=True)
for e in rr.sample(TREINO + PROVA, min(30, len(TREINO + PROVA))):
    print(f"\n[nota {e['nota_juiz']} | {e['fonte']} | {e['tema']}]", flush=True); mostrar(e)
print("\n===== 10 reprovadas pelo juiz =====", flush=True)
for e in rr.sample(FORA, min(10, len(FORA))):
    print(f"\n[nota {e['nota_juiz']} | {e['fonte']}]", flush=True); mostrar(e)
print("\nPRONTO. Leia as conversas acima e mande o print para o assistente.", flush=True)